In [1]:
from ingest import load_lessons_data
documents = load_lessons_data()

In [2]:
documents[10]

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

In [3]:
documents_llm = []

for i, doc in enumerate(documents):
    doc_with_id = dict(doc)
    doc_with_id["id"] = i
    documents_llm.append(doc_with_id)

len(documents_llm)

72

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["filename"])
print(doc["content"])

0
01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple 

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [10]:
import json
user_prompt = json.dumps({
    "id": doc["id"],
    "filename": doc["filename"],
    "content": doc["content"]
})

In [11]:
user_prompt

'{"id": 0, "filename": "01-agentic-rag/lessons/01-intro.md", "content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"yo

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.chat.completions.parse(
    model="gemini-3.1-flash-lite",
    messages=messages,
    response_format=Questions
)

In [14]:
response.choices[0].message.parsed.questions

['Why are we building these systems using raw Python instead of just using existing frameworks?',
 'How does the way an LLM operates compare to the predictive text feature on a smartphone?',
 'What are the main drawbacks I should be aware of when relying on large language models?',
 'Why is Retrieval-Augmented Generation considered a better approach than just relying on what a model already knows?',
 'What is the key difference between the pipeline we build in the first part versus the agentic version in the second part?']

In [15]:
response.usage.prompt_tokens

1070

Question 1

In [16]:
usages = []

for doc in documents:
    if doc["filename"] == "01-agentic-rag/lessons/01-intro.md" or doc["filename"] == "01-agentic-rag/lessons/02-environment.md" or doc["filename"] == "01-agentic-rag/lessons/03-rag.md":
        print(doc["id"])
        print(doc["filename"])

        user_prompt = json.dumps({
            "id": doc["id"],
            "filename": doc["filename"],
            "content": doc["content"]
        })

        messages = [
            {"role": "developer", "content": data_gen_instructions},
            {"role": "user", "content": user_prompt}
        ]

        response = openai_client.chat.completions.parse(
            model="gemini-3.1-flash-lite",
            messages=messages,
            response_format=Questions
        )

        print(response.choices[0].message.parsed.questions)
        print(response.usage.prompt_tokens)
        usages.append(response.usage.prompt_tokens)

0
01-agentic-rag/lessons/01-intro.md
['Why are we manually building a search index instead of using existing tools?', 'How does a basic language model differ from these huge neural networks we use?', 'What are the main issues I should worry about when relying on LLMs for information?', "Why is RAG considered better than just relying on the model's training knowledge?", 'What exactly is the difference between the pipeline we build in the first part and the agentic version in the second?']
1070
1
01-agentic-rag/lessons/02-environment.md
["What specific tool should I use to manage the project's dependencies?", 'Can I use services other than OpenAI for this class?', 'Is it okay to push my credentials file to a public repository?', 'What is the recommended method to avoid calling load_dotenv inside every single notebook?', 'How much money do I need to put into my OpenAI account before I can start?']
1407
2
01-agentic-rag/lessons/03-rag.md
["Why can't I just ask an LLM directly about specifi

In [17]:
sum(usages) / len(usages)

1456.6666666666667

Question 2

In [18]:
import pandas as pd

In [19]:
df_ground_truth = pd.read_csv('ground-truth.csv')

In [20]:
ground_truth = df_ground_truth.to_dict(orient="records")


In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [22]:
from ingest import build_index

In [23]:
index = build_index(chunks)

In [24]:
def text_search(query, num_results = 5):
    boost_dict = {"content": 3.0, "start": 0.5}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

In [25]:
from embedder import Embedder

embed = Embedder()

2026-07-13 16:28:57.547455324 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [26]:
texts = []

for chunk in chunks:
    text = chunk["filename"] + " " + chunk["content"]
    texts.append(text)

In [27]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = [embed.encode(text) for text in batch]
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

295

In [28]:
import numpy as np
X = np.array(vectors)

In [29]:

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, chunks)

In [30]:
def vector_search(query, num_results = 5):
    query_vectored = embed.encode(query)

    return vindex.search(
        query_vectored,
        num_results=num_results,
    )

In [31]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [32]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [33]:
q = ground_truth[0]["question"]

In [34]:
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

Question 3

In [35]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

Question 4

In [50]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [38]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [53]:
relevance_total = compute_relevance_total(ground_truth, text_search)


  0%|          | 0/360 [00:00<?, ?it/s]

In [54]:
relevance_total

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 1, 1, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 1, 0],
 [1, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0,

In [55]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [56]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [58]:
evaluate(ground_truth, text_search)


  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

Question 5

In [59]:
evaluate(ground_truth, vector_search)


  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7277777777777777, 'mrr': 0.5469907407407407}

Question 6

In [61]:
def compute_relevance(q, search_function, k):
    doc_id = q["filename"]
    results = search_function(query=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [62]:
def compute_relevance_total(ground_truth, search_function, k):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, k = k)
        relevance_total.append(relevance)

    return relevance_total

In [63]:
def evaluate(ground_truth, search_function, k):
    relevance_total = compute_relevance_total(ground_truth, search_function, k)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [71]:
data_list = []

for k in [1, 50, 100, 200]:  
    results = evaluate(ground_truth, hybrid_search, k)
    data_list.append({
        'k': k,
        'hit_rate': results["hit_rate"],
        'mrr': results["mrr"]
    }) 


  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [72]:
df = pd.DataFrame(data_list)

In [79]:
df.sort_values(by='mrr', inplace=True, ascending=False)

In [80]:
df

,k,hit_rate,mrr
0,1,0.819444,0.648611
3,200,0.836111,0.641898
2,100,0.836111,0.641898
1,50,0.836111,0.641898
